# UTTA-Med — Gate D (MC-Dropout + gates). NO retrain.

GPU T4, Internet on. **Add Input** `camelyon17_resnet18_source_s42_BEST.pt`.

Frozen from Gate C: `bn_freeze_stats`, Adam **lr=1e-5**, continual 1-step, shuffle=False.

τ selected on hospital 1 only. Test hospital 2 once at the end.

In [ ]:
!pip -q install datasets pyarrow scikit-learn tqdm pillow
import torch
assert torch.cuda.is_available(), 'Enable GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
import json, random, math
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm
from datasets import load_dataset

SEED, LR, N_MC, BATCH = 42, 1e-5, 20, 64
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda')

def find_ckpt():
    for p in list(Path('/kaggle/input').rglob('*.pt')) + list(WORK.glob('*.pt')):
        try:
            st = torch.load(p, map_location='cpu', weights_only=False)
            print('found', p, 'epoch', st.get('epoch'), 'val_auroc', st.get('val_auroc'))
            if int(st.get('epoch', -1))==3 and float(st.get('val_auroc',0))>=0.978:
                return p, st
        except Exception as e:
            print('skip', p, e)
    raise FileNotFoundError('Add Input BEST epoch-3 ckpt')
CKPT_PATH, CKPT = find_ckpt()
print('USING', CKPT_PATH)

In [ ]:
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf = transforms.Compose([transforms.Resize((96,96)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
def cid(v): return int(v.item() if hasattr(v,'item') else v)
class HFCamelyon(Dataset):
    def __init__(self, split, centers):
        centers={int(c) for c in centers}
        keep=[i for i,c in enumerate(split['center']) if cid(c) in centers]
        self.ds = split.select(keep) if len(keep)<len(split) else split
        print(' kept', len(self.ds), sorted(centers))
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        r=self.ds[i]; img=r['image']
        img = img.convert('RGB') if isinstance(img, Image.Image) else Image.fromarray(np.array(img)).convert('RGB')
        return tf(img), torch.tensor(float(int(r['label'])))
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        bb=models.resnet18(weights=None); d=bb.fc.in_features; bb.fc=nn.Identity(); self.backbone=bb
        self.classifier=nn.Sequential(nn.Linear(d,d), nn.ReLU(True), nn.Dropout(0.5), nn.Linear(d,1))
    def forward(self,x): return self.classifier(self.backbone(x))

raw=load_dataset('wltjr1007/Camelyon17-WILDS')
ood_set=HFCamelyon(raw['validation'],{1})
test_set=HFCamelyon(raw['test'],{2})
ood_loader=DataLoader(ood_set,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)
test_loader=DataLoader(test_set,batch_size=BATCH,shuffle=False,num_workers=0,pin_memory=True)

In [ ]:
def ece_score(y,p,n=15):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); bins=np.linspace(0,1,n+1); e=0.0
    for i in range(n):
        m=(p>bins[i])&(p<=bins[i+1]) if i else (p>=bins[i])&(p<=bins[i+1])
        if m.any(): e += m.mean()*abs(y[m].mean()-p[m].mean())
    return float(e)
def metrics_of(y,p):
    y,p=np.asarray(y).ravel(),np.asarray(p).ravel(); pred=(p>=0.5).astype(int)
    return dict(n=int(len(y)), accuracy=float(accuracy_score(y,pred)),
        precision=float(precision_score(y,pred,zero_division=0)), recall=float(recall_score(y,pred,zero_division=0)),
        sensitivity=float(recall_score(y,pred,zero_division=0)),
        specificity=float(((y==0)&(pred==0)).sum()/max((y==0).sum(),1)),
        f1=float(f1_score(y,pred,zero_division=0)), auroc=float(roc_auc_score(y,p)),
        auprc=float(average_precision_score(y,p)), ece=ece_score(y,p), brier=float(np.mean((p-y)**2)))
def flip_table(y,b,a):
    y=np.asarray(y).ravel().astype(int); b=(np.asarray(b).ravel()>=0.5).astype(int); a=(np.asarray(a).ravel()>=0.5).astype(int)
    return dict(correction_rate=float(((b!=y)&(a==y)).sum()/max((b!=y).sum(),1)),
                harm_rate=float(((b==y)&(a!=y)).sum()/max((b==y).sum(),1)), flip_rate=float((a!=b).mean()),
                correction=int(((b!=y)&(a==y)).sum()), harmful=int(((b==y)&(a!=y)).sum()))
def load_fresh():
    m=Net().to(device); m.load_state_dict(CKPT['model']); return m
def bn_affine(m):
    for p in m.parameters(): p.requires_grad=False
    ps=[]
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d) and mod.affine:
            mod.weight.requires_grad=True; mod.bias.requires_grad=True; ps += [mod.weight, mod.bias]
    return ps
def eval_mode_frozen_bn(m):
    m.eval()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()
def mc_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d): mod.eval()
        elif isinstance(mod, nn.Dropout): mod.train()
def tta_mode(m):
    m.train()
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.eval()
        elif isinstance(mod, nn.BatchNorm2d): mod.eval()
def entropy(z):
    p=torch.sigmoid(z).clamp(1e-6,1-1e-6)
    return -(p*torch.log(p)+(1-p)*torch.log(1-p)).view(-1)
@torch.no_grad()
def mc_predict(m, x, n=N_MC):
    mc_mode(m); ps=[]
    for _ in range(n):
        ps.append(torch.sigmoid(m(x)).view(-1))
    P=torch.stack(ps,0)
    mean=P.mean(0); var=P.var(0, unbiased=False)
    return mean, var
@torch.no_grad()
def evaluate(m, ld):
    eval_mode_frozen_bn(m); ys,ps=[],[]
    for x,y in ld:
        x=x.to(device); ps.append(torch.sigmoid(m(x)).view(-1).cpu().numpy()); ys.append(y.view(-1).numpy())
    return metrics_of(np.concatenate(ys), np.concatenate(ps))

src=load_fresh()
ood_src=evaluate(src, ood_loader)
print('ASSERT OOD AUROC', round(ood_src['auroc'],4))
assert ood_src['auroc']>=0.975, 'wrong ckpt'

In [ ]:
print('--- Gate D.1  U vs error on OOD val (must see U up => error up) ---')
src=load_fresh(); ys, us, ps, ents, confs = [], [], [], [], []
for x,y in tqdm(ood_loader, desc='mc val'):
    x=x.to(device)
    mean, var = mc_predict(src, x, N_MC)
    p=mean.clamp(1e-6,1-1e-6)
    ys.append(y.view(-1).numpy()); us.append(var.cpu().numpy()); ps.append(mean.cpu().numpy())
    ents.append((-(p*torch.log(p)+(1-p)*torch.log(1-p))).cpu().numpy())
    confs.append(torch.maximum(p,1-p).cpu().numpy())
y=np.concatenate(ys); U=np.concatenate(us); P=np.concatenate(ps); H=np.concatenate(ents); C=np.concatenate(confs)
err=( (P>=0.5).astype(int) != y.astype(int) ).astype(float)
order=np.argsort(U); nbin=10; edges=np.array_split(order, nbin)
u_bins=[]
print('bin  n   meanU     err_rate')
for i, idx in enumerate(edges):
    row=dict(bin=i+1, n=int(len(idx)), mean_U=float(U[idx].mean()), err_rate=float(err[idx].mean()))
    u_bins.append(row)
    print(f"{row['bin']:3d} {row['n']:5d} {row['mean_U']:.6f}  {row['err_rate']:.4f}")
corr=float(np.corrcoef(U, err)[0,1])
print('Pearson corr(U, error)=', round(corr,4))
if corr < 0.02:
    print('WARNING: weak U-error link. Still complete tau sweep; do not over-claim.')

In [ ]:
print('--- tau sweep on OOD val (percentiles of U from frozen source MC-Dropout) ---')

def adapt_utta(ld, tau):
    m=load_fresh(); srcm=load_fresh(); eval_mode_frozen_bn(srcm)
    opt=torch.optim.Adam(bn_affine(m), lr=LR)
    n_seen=n_kept=n_skip_batch=0; ys,pb,pa=[],[],[]
    for x,y in tqdm(ld, desc=f'utta tau={tau:.6g}'):
        x=x.to(device); B=x.size(0); n_seen += B
        with torch.no_grad():
            eval_mode_frozen_bn(srcm)
            pb.append(torch.sigmoid(srcm(x)).view(-1).cpu().numpy())
            _, var = mc_predict(srcm, x, N_MC)  # frozen source U — not the adapting model
            w = (var < tau).float()
        tta_mode(m)
        k=int(w.sum().item())
        if k==0:
            n_skip_batch += 1
        else:
            opt.zero_grad(set_to_none=True)
            loss=(w*entropy(m(x))).sum()/w.sum()
            loss.backward(); opt.step(); n_kept += k
        with torch.no_grad():
            eval_mode_frozen_bn(m); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y,b,a=np.concatenate(ys),np.concatenate(pb),np.concatenate(pa)
    return dict(gate='utta', tau=float(tau), lr=LR, n_mc=N_MC, coverage=n_kept/max(n_seen,1),
                n_skip_batch=n_skip_batch, before=metrics_of(y,b), after=metrics_of(y,a), harm=flip_table(y,b,a))

pcts=[20,30,40,50,60,70,80]
taus=[float(np.quantile(U, p/100)) for p in pcts]
print('tau percentiles', list(zip(pcts, [round(t,8) for t in taus])))
tau_rows=[]
for pct,t in zip(pcts, taus):
    r=adapt_utta(ood_loader, t); r['percentile']=pct; r['split']='ood_val'; tau_rows.append(r)
    print('p', pct, 'tau', f'{t:.6g}', 'cov', round(r['coverage'],3),
          'AUROC', round(r['after']['auroc'],4), 'F1', round(r['after']['f1'],4),
          'ECE', round(r['after']['ece'],4), 'harm', round(r['harm']['harm_rate'],4),
          'skip_batches', r['n_skip_batch'])

# Predefined selection: max val AUROC; tie-break lowest harm.
best=max(tau_rows, key=lambda r: (r['after']['auroc'], -r['harm']['harm_rate']))
print('SELECTED tau', best['tau'], 'percentile', best['percentile'], 'cov', round(best['coverage'],3),
      'val AUROC', round(best['after']['auroc'],4))
target_cov=float(best['coverage'])


In [ ]:
print('--- matched-coverage thresholds on OOD val ---')
# confidence: accept if conf > t, choose t so coverage ~= target_cov
conf_t = float(np.quantile(C, 1.0-target_cov))
ent_t = float(np.quantile(H, target_cov))
print('target_cov', round(target_cov,3), 'conf_t', conf_t, 'ent_t', ent_t)
print('empirical conf coverage', float((C>conf_t).mean()), 'ent coverage', float((H<ent_t).mean()))

def adapt_simple(ld, gate, tau=None, target_cov=None, rng_seed=SEED):
    m=load_fresh(); srcm=load_fresh(); eval_mode_frozen_bn(srcm)
    opt=torch.optim.Adam(bn_affine(m), lr=LR)
    rng=np.random.RandomState(rng_seed)
    n_seen=n_kept=n_skip_batch=0; ys,pb,pa=[],[],[]
    for x,y in tqdm(ld, desc=gate):
        x=x.to(device); B=x.size(0); n_seen += B
        with torch.no_grad():
            eval_mode_frozen_bn(srcm); p0=torch.sigmoid(srcm(x)).view(-1); pb.append(p0.cpu().numpy())
            if gate=='tent':
                w=torch.ones(B, device=x.device)
            elif gate=='random':
                w=torch.tensor(rng.rand(B)<target_cov, device=x.device, dtype=torch.float32)
            elif gate=='confidence':
                w=(torch.maximum(p0,1-p0)>tau).float()
            elif gate=='entropy':
                w=(entropy(torch.logit(p0.clamp(1e-6,1-1e-6)))<tau).float()
            else:
                raise ValueError(gate)
        tta_mode(m); k=int(w.sum().item())
        if k==0:
            n_skip_batch += 1
        else:
            opt.zero_grad(set_to_none=True)
            loss=(w*entropy(m(x))).sum()/w.sum(); loss.backward(); opt.step(); n_kept += k
        with torch.no_grad():
            eval_mode_frozen_bn(m); pa.append(torch.sigmoid(m(x)).view(-1).cpu().numpy())
        ys.append(y.view(-1).numpy())
    y,b,a=np.concatenate(ys),np.concatenate(pb),np.concatenate(pa)
    return dict(gate=gate, tau=None if tau is None else float(tau), target_cov=target_cov, lr=LR,
                coverage=n_kept/max(n_seen,1), n_skip_batch=n_skip_batch,
                before=metrics_of(y,b), after=metrics_of(y,a), harm=flip_table(y,b,a))

print('--- TEST once ---')
src_test=evaluate(load_fresh(), test_loader)
tent_test=adapt_simple(test_loader,'tent'); tent_test['split']='test'
rand_test=adapt_simple(test_loader,'random', target_cov=target_cov); rand_test['split']='test'
conf_test=adapt_simple(test_loader,'confidence', tau=conf_t); conf_test['split']='test'
ent_test=adapt_simple(test_loader,'entropy', tau=ent_t); ent_test['split']='test'
utta_test=adapt_utta(test_loader, best['tau']); utta_test['split']='test'

def line(name, r, src=False):
    m = r if src else r['after']
    cov = 0 if src else r.get('coverage')
    harm = None if src else r['harm']['harm_rate']
    print(f"{name:12s} AUROC={m['auroc']:.4f} F1={m['f1']:.4f} ECE={m['ece']:.4f} cov={cov if src else round(cov,3)} harm={harm if src else round(harm,4)}")

print('=== GATE D TEST ===')
line('source', src_test, src=True)
line('tent', tent_test)
line('random', rand_test)
line('confidence', conf_test)
line('entropy', ent_test)
line('utta', utta_test)

out={
    'cuda': True, 'device': torch.cuda.get_device_name(0),
    'checkpoint': str(CKPT_PATH), 'lr': LR, 'n_mc': N_MC, 'bn_mode': 'bn_freeze_stats',
    'tau_selected': best['tau'], 'tau_percentile': best['percentile'], 'matched_coverage': target_cov,
    'conf_threshold': conf_t, 'entropy_threshold': ent_t,
    'u_error_corr': corr, 'u_bins': u_bins,
    'val_tau_sweep': tau_rows, 'source_ood': ood_src, 'source_test': src_test,
    'tent_test': tent_test, 'random_test': rand_test, 'confidence_test': conf_test,
    'entropy_test': ent_test, 'utta_test': utta_test,
    'selection_rule': 'max val AUROC then min harm; matched coverage for random/conf/entropy',
}
p=WORK/'camelyon17_resnet18_utta_gated_s42.json'
p.write_text(json.dumps(out, indent=2))
print('wrote', p)